# Notebook 3: SFT Training on Kaggle (T4 × 2)

This notebook runs **Supervised Fine-Tuning (SFT)** on GQA using **DistributedDataParallel (DDP)** across two Kaggle T4 GPUs.

### Training condition
- Input: **full image + question**
- Target: **ground-truth GQA answer**
- Loss: supervised cross-entropy
- Fine-tuning: **LoRA**
- Evaluation: **full image only**
- **Primary metric:** BERTScore-F1 (`roberta-large`)
- **Secondary metric:** Normalised exact-match accuracy
- Parallelism: **2 processes / 2 GPUs via `torchrun`**

### Kaggle dataset structure expected

Attach one Kaggle dataset containing:

```text
data/
├── processed/
│   ├── train.csv
│   ├── val.csv
│   ├── test.csv
│   └── all_gqa.csv
├── images/
│   └── <image_id>.jpg
└── crops/
    └── <qa_id>.jpg
```

Only `processed/` and `images/` are used by this SFT notebook. The crop folder is included only because Notebook 4 will need it.

> **Important:** Enable **GPU T4 ×2** in Kaggle before running this notebook.

3 epochs are run. After every epoch it runs validation, computes **BERTScore-F1** as the primary metric, records normalised exact match as a secondary metric, saves the epoch adapter, and updates `best/` whenever validation BERTScore-F1 improves.


## Install dependencies

In [19]:
%pip uninstall -y torchaudio torchao

%pip install -q -U \
    "transformers>=5.7,<6" \
    "peft>=0.17" \
    "accelerate>=1.0" \
    torchvision \
    pandas \
    Pillow \
    tqdm \
    bottleneck \
    bert-score


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Verify Kaggle sees both T4 GPUs

In [20]:
import os
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"{props.total_memory / 1024**3:.1f} GB"
    )

if torch.cuda.device_count() != 2:
    raise RuntimeError(
        "This notebook expects Kaggle GPU T4 x2. "
        "Open Notebook settings and select GPU T4 x2."
    )


CUDA available: True
GPU count: 2
GPU 0: Tesla T4 | 14.6 GB
GPU 1: Tesla T4 | 14.6 GB


## Configure Kaggle paths and experiment settings

Change `KAGGLE_DATASET_SLUG` to the folder name shown under `/kaggle/input/`.

For example, if Kaggle mounts your dataset as:

```text
/kaggle/input/cos802-gqa-data
```

then use:

```python
KAGGLE_DATASET_SLUG = "cos802-gqa-data"
```


In [21]:
from pathlib import Path

# ============================================================
# KAGGLE DATASET
# ============================================================

KAGGLE_DATASET_SLUG = "cos802-gqa-data-v2"

KAGGLE_INPUT = Path(
    "/kaggle/input/datasets/vishalbhagauni/"
) / KAGGLE_DATASET_SLUG

DATA = KAGGLE_INPUT
PROCESSED = DATA / "processed"
IMAGES = DATA / "images"
CROPS = DATA / "crops"

# Writable Kaggle storage
WORKING = Path("/kaggle/working")
RESULTS = WORKING / "results"
MODELS_DIR = WORKING / "models"

RESULTS.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# EXPERIMENT
# ============================================================

MODEL_KEY = "smolvlm"
# "smolvlm", "blip", or "minicpm"

SEED = 42

EPOCHS = 3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01

# This is per GPU.
PER_DEVICE_BATCH_SIZE = 1

# Effective batch size =
# PER_DEVICE_BATCH_SIZE × WORLD_SIZE × GRAD_ACCUM_STEPS
GRAD_ACCUM_STEPS = 4

MAX_GRAD_NORM = 1.0

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

MAX_NEW_TOKENS = 24

# Smoke test first with e.g. 64 / 64.
TRAIN_LIMIT = None
VAL_LIMIT = None

SAVE_EVERY_OPT_STEPS = 50

print("Dataset:", KAGGLE_INPUT)
print("Processed:", PROCESSED)
print("Images:", IMAGES)
print("Results:", RESULTS)
print("Models:", MODELS_DIR)


Dataset: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data-v2
Processed: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data-v2/processed
Images: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data-v2/images
Results: /kaggle/working/results
Models: /kaggle/working/models


## Validate attached data

In [22]:
import pandas as pd

TRAIN_CSV = PROCESSED / "train.csv"
VAL_CSV = PROCESSED / "val.csv"

for path in [TRAIN_CSV, VAL_CSV]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )

train_check = pd.read_csv(
    TRAIN_CSV,
    nrows=5,
    keep_default_na=False,
)

val_check = pd.read_csv(
    VAL_CSV,
    nrows=5,
    keep_default_na=False,
)

print("Train columns:")
print(train_check.columns.tolist())

print("\nExample local Kaggle image path:")
example_image = (
    IMAGES
    / f"{train_check.iloc[0]['image_id']}.jpg"
)

print(example_image)
print("Exists:", example_image.is_file())

if not example_image.is_file():
    raise FileNotFoundError(
        "The image directory does not match the expected Kaggle structure. "
        "Check KAGGLE_DATASET_SLUG and the uploaded data layout."
    )


Train columns:
['qa_id', 'image_id', 'question', 'answer', 'full_answer', 'source_split', 'structural_type', 'semantic_type', 'detailed_type', 'semantic_program', 'program_operations', 'program_base_operations', 'program_arguments', 'grounded_object_ids', 'grounded_object_names', 'scene_width', 'scene_height', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'crop_area_ratio', 'n_grounded_objects', 'crop_filter_reason', 'image_path', 'crop_path', 'crop_x1', 'crop_y1', 'crop_x2', 'crop_y2', 'actual_width', 'actual_height', 'final_crop_area_ratio', 'split']

Example local Kaggle image path:
/kaggle/input/datasets/vishalbhagauni/cos802-gqa-data-v2/images/2404416.jpg
Exists: True


## Write the two-GPU DDP training script

The script is launched as:

```bash
torchrun --standalone --nproc_per_node=2 train_sft_ddp.py
```

Each T4 receives a different shard of the training set through `DistributedSampler`. Gradients are synchronized automatically by PyTorch DDP.


In [23]:
TRAIN_SCRIPT = r"""
import os
import json
import random
import re
import string
import gc
import shutil
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from PIL import Image
from tqdm.auto import tqdm
from bert_score import BERTScorer

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BlipProcessor,
    BlipForQuestionAnswering,
)

from peft import (
    LoraConfig,
    get_peft_model,
)


# ============================================================
# CONFIG FROM ENVIRONMENT
# ============================================================

KAGGLE_DATASET_SLUG = os.environ["KAGGLE_DATASET_SLUG"]

DATA = Path(
    "/kaggle/input/datasets/vishalbhagauni/"
) / KAGGLE_DATASET_SLUG

PROCESSED = DATA / "processed"
IMAGES = DATA / "images"

WORKING = Path("/kaggle/working")
RESULTS = WORKING / "results"
MODELS_DIR = WORKING / "models"

MODEL_KEY = os.environ.get(
    "MODEL_KEY",
    "smolvlm",
)

SEED = int(
    os.environ.get(
        "SEED",
        "42",
    )
)

EPOCHS = int(
    os.environ.get(
        "EPOCHS",
        "3",
    )
)

LEARNING_RATE = float(
    os.environ.get(
        "LEARNING_RATE",
        "1e-4",
    )
)

WEIGHT_DECAY = float(
    os.environ.get(
        "WEIGHT_DECAY",
        "0.01",
    )
)

PER_DEVICE_BATCH_SIZE = int(
    os.environ.get(
        "PER_DEVICE_BATCH_SIZE",
        "1",
    )
)

GRAD_ACCUM_STEPS = int(
    os.environ.get(
        "GRAD_ACCUM_STEPS",
        "4",
    )
)

MAX_GRAD_NORM = float(
    os.environ.get(
        "MAX_GRAD_NORM",
        "1.0",
    )
)

LORA_R = int(
    os.environ.get(
        "LORA_R",
        "8",
    )
)

LORA_ALPHA = int(
    os.environ.get(
        "LORA_ALPHA",
        "16",
    )
)

LORA_DROPOUT = float(
    os.environ.get(
        "LORA_DROPOUT",
        "0.05",
    )
)

MAX_NEW_TOKENS = int(
    os.environ.get(
        "MAX_NEW_TOKENS",
        "24",
    )
)

SAVE_EVERY_OPT_STEPS = int(
    os.environ.get(
        "SAVE_EVERY_OPT_STEPS",
        "50",
    )
)

TRAIN_LIMIT_RAW = os.environ.get(
    "TRAIN_LIMIT",
    "",
)

VAL_LIMIT_RAW = os.environ.get(
    "VAL_LIMIT",
    "",
)

TRAIN_LIMIT = (
    int(TRAIN_LIMIT_RAW)
    if TRAIN_LIMIT_RAW
    else None
)

VAL_LIMIT = (
    int(VAL_LIMIT_RAW)
    if VAL_LIMIT_RAW
    else None
)


MODEL_IDS = {
    "smolvlm": "HuggingFaceTB/SmolVLM-500M-Instruct",
    "blip": "Salesforce/blip-vqa-base",
    "minicpm": "openbmb/MiniCPM-V-4.6",
}

MODEL_ID = MODEL_IDS[
    MODEL_KEY
]


# ============================================================
# DDP INITIALIZATION
# ============================================================

LOCAL_RANK = int(
    os.environ["LOCAL_RANK"]
)

torch.cuda.set_device(
    LOCAL_RANK
)

DEVICE = torch.device(
    f"cuda:{LOCAL_RANK}"
)

dist.init_process_group(
    backend="nccl",
    device_id=DEVICE,
    timeout=timedelta(hours=1),
)

RANK = dist.get_rank()
WORLD_SIZE = dist.get_world_size()

MODEL_DTYPE = torch.float16


def is_main():
    # Only rank 0 handles shared console output and filesystem writes.
    return RANK == 0


def rank0_print(*args, **kwargs):
    # Avoid duplicated log messages because every DDP process runs this script.
    if is_main():
        print(
            *args,
            **kwargs,
            flush=True,
        )


# ============================================================
# REPRODUCIBILITY
# ============================================================

rank_seed = (
    SEED
    + RANK
)

random.seed(
    rank_seed
)

np.random.seed(
    rank_seed
)

torch.manual_seed(
    rank_seed
)

torch.cuda.manual_seed_all(
    rank_seed
)


# ============================================================
# OUTPUT PATHS
# ============================================================

RUN_DIR = (
    RESULTS
    / "training"
    / "sft"
    / MODEL_KEY
    / f"seed{SEED}"
)

ADAPTER_DIR = (
    MODELS_DIR
    / "sft"
    / MODEL_KEY
    / f"seed{SEED}"
)

if is_main():
    RUN_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    ADAPTER_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

dist.barrier()


# ============================================================
# LOAD DATA
# ============================================================

train_df = pd.read_csv(
    PROCESSED / "train.csv",
    keep_default_na=False,
)

val_df = pd.read_csv(
    PROCESSED / "val.csv",
    keep_default_na=False,
)


# Replace Mac paths stored in CSVs with Kaggle paths.
def remap_paths(df):
    # The CSVs are portable metadata; image files live at the Kaggle path.
    df = df.copy()

    df["image_path"] = (
        df["image_id"]
        .astype(str)
        .map(
            lambda x:
                str(
                    IMAGES
                    / f"{x}.jpg"
                )
        )
    )

    return df


train_df = remap_paths(
    train_df
)

val_df = remap_paths(
    val_df
)


if TRAIN_LIMIT is not None:
    train_df = (
        train_df
        .sample(
            n=min(
                TRAIN_LIMIT,
                len(train_df),
            ),
            random_state=SEED,
        )
        .reset_index(
            drop=True
        )
    )

if VAL_LIMIT is not None:
    val_df = (
        val_df
        .sample(
            n=min(
                VAL_LIMIT,
                len(val_df),
            ),
            random_state=SEED,
        )
        .reset_index(
            drop=True
        )
    )


rank0_print(
    "World size:",
    WORLD_SIZE,
)

rank0_print(
    "Train examples:",
    len(train_df),
)

rank0_print(
    "Validation examples:",
    len(val_df),
)

rank0_print(
    "Effective batch size:",
    PER_DEVICE_BATCH_SIZE
    * WORLD_SIZE
    * GRAD_ACCUM_STEPS,
)


# ============================================================
# DATASET
# ============================================================

class GQADataset(Dataset):
    # Keep dataset items as lightweight metadata until each rank preprocesses them.

    def __init__(
        self,
        df,
    ):
        self.df = (
            df
            .reset_index(
                drop=True
            )
        )

    def __len__(
        self,
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        idx,
    ):
        row = self.df.iloc[
            idx
        ]

        return {
            "qa_id":
                str(
                    row.qa_id
                ),

            "image_path":
                str(
                    row.image_path
                ),

            "question":
                str(
                    row.question
                ),

            "answer":
                str(
                    row.answer
                ),
        }


def collate_rows(
    batch,
):
    # We intentionally keep raw Python rows.
    # VLM preprocessing occurs inside each process.
    return batch


train_dataset = GQADataset(
    train_df
)

train_sampler = DistributedSampler(
    train_dataset,
    num_replicas=WORLD_SIZE,
    rank=RANK,
    shuffle=True,
    seed=SEED,
    drop_last=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=PER_DEVICE_BATCH_SIZE,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=collate_rows,
)


# ============================================================
# LOAD MODEL
# ============================================================

rank0_print(
    "Loading model:",
    MODEL_ID,
)

if MODEL_KEY == "blip":

    processor = (
        BlipProcessor
        .from_pretrained(
            MODEL_ID
        )
    )

    base_model = (
        BlipForQuestionAnswering
        .from_pretrained(
            MODEL_ID,
            dtype=MODEL_DTYPE,
        )
    )

else:

    processor = (
        AutoProcessor
        .from_pretrained(
            MODEL_ID
        )
    )

    base_model = (
        AutoModelForImageTextToText
        .from_pretrained(
            MODEL_ID,
            dtype=MODEL_DTYPE,
            low_cpu_mem_usage=True,
        )
    )


# ============================================================
# GRADIENT CHECKPOINTING
# ============================================================

# Disable for BLIP because it is not needed here and can
# interact badly with the PEFT/autograd path.
if MODEL_KEY != "blip":

    if hasattr(
        base_model,
        "gradient_checkpointing_enable",
    ):
        try:
            base_model.gradient_checkpointing_enable()
        except Exception:
            pass


if hasattr(
    base_model.config,
    "use_cache",
):
    base_model.config.use_cache = False


# ============================================================
# MODEL-SPECIFIC LORA TARGETS
# ============================================================

if MODEL_KEY == "blip":

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=[
            "query",
            "value",
        ],
    )

else:

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules="all-linear",
    )


base_model = get_peft_model(
    base_model,
    lora_config,
)

base_model = get_peft_model(
    base_model,
    lora_config,
)


base_model = base_model.to(
    DEVICE
)


# ============================================================
# DDP
# ============================================================

FIND_UNUSED_PARAMETERS = (
    MODEL_KEY == "minicpm"
)

model = DDP(
    base_model,
    device_ids=[
        LOCAL_RANK
    ],
    output_device=LOCAL_RANK,
    find_unused_parameters=FIND_UNUSED_PARAMETERS,
)

if is_main():
    base_model.print_trainable_parameters()


# ============================================================
# HELPERS
# ============================================================

def open_rgb(
    path,
):
    with Image.open(
        path
    ) as im:

        return im.convert(
            "RGB"
        )


def move_batch(
    batch,
):
    # Move tensors to this rank's GPU while preserving integer token tensors.
    moved = {}

    for key, value in batch.items():

        if not torch.is_tensor(
            value
        ):
            moved[
                key
            ] = value

            continue


        if (
            key
            in {
                "pixel_values",
                "pixel_values_videos",
            }
            and value.is_floating_point()
        ):

            moved[
                key
            ] = value.to(
                device=DEVICE,
                dtype=MODEL_DTYPE,
                non_blocking=True,
            )

        else:

            moved[
                key
            ] = value.to(
                DEVICE,
                non_blocking=True,
            )

    return moved


def generic_chat_kwargs():
    # Keep model-specific processor options in one place for training and evaluation.

    if MODEL_KEY == "minicpm":

        return {
            "enable_thinking":
                False,

            "processor_kwargs":
                {
                    "downsample_mode":
                        "4x",

                    "max_slice_nums":
                        1,
                },
        }

    return {}


# ============================================================
# SINGLE-EXAMPLE SFT INPUT
# ============================================================

def prepare_example(
    row,
):
    # Build labels so loss is calculated only for the answer, not the prompt.

    image = open_rgb(
        row[
            "image_path"
        ]
    )

    question = row[
        "question"
    ]

    answer = row[
        "answer"
    ]


    if MODEL_KEY == "blip":

        inputs = processor(
            images=image,
            text=question,
            return_tensors="pt",
        )

        answer_tokens = (
            processor.tokenizer(
                answer,
                return_tensors="pt",
                add_special_tokens=True,
            )[
                "input_ids"
            ]
        )

        inputs[
            "labels"
        ] = answer_tokens

        return move_batch(
            inputs
        )


    prompt_messages = [
        {
            "role":
                "user",

            "content":
                [
                    {
                        "type":
                            "image",

                        "image":
                            image,
                    },
                    {
                        "type":
                            "text",

                        "text":
                            question
                            + "\\nAnswer briefly with only the answer.",
                    },
                ],
        }
    ]


    full_messages = [
        *prompt_messages,
        {
            "role":
                "assistant",

            "content":
                [
                    {
                        "type":
                            "text",

                        "text":
                            answer,
                    }
                ],
        },
    ]


    kwargs = (
        generic_chat_kwargs()
    )


    prompt_inputs = (
        processor
        .apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            **kwargs,
        )
    )


    full_inputs = (
        processor
        .apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_tensors="pt",
            **kwargs,
        )
    )


    prompt_len = (
        prompt_inputs[
            "input_ids"
        ]
        .shape[
            1
        ]
    )


    labels = (
        full_inputs[
            "input_ids"
        ]
        .clone()
    )


    labels[
        :,
        :prompt_len,
    ] = -100


    full_inputs[
        "labels"
    ] = labels


    return move_batch(
        full_inputs
    )


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# TRAINING + PER-EPOCH VALIDATION
# ============================================================

ARTICLES = re.compile(
    r"\b(a|an|the)\b"
)

PUNCT = str.maketrans(
    "",
    "",
    string.punctuation,
)


def normalize_answer(x):
    # Match the usual VQA scoring convention by ignoring case, punctuation, and articles.
    text = (
        str(x)
        .lower()
        .translate(PUNCT)
    )

    text = ARTICLES.sub(
        " ",
        text,
    )

    return " ".join(
        text.split()
    )


@torch.inference_mode()
def predict_answer(
    image_path,
    question,
):
    # Validation uses the unwrapped model because no DDP gradient synchronization is needed.
    eval_model = model.module
    eval_model.eval()

    image = open_rgb(
        image_path
    )

    if MODEL_KEY == "blip":
        inputs = processor(
            images=image,
            text=question,
            return_tensors="pt",
        )

        inputs = move_batch(
            inputs
        )

        generated = eval_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
        )

        return (
            processor
            .decode(
                generated[0],
                skip_special_tokens=True,
            )
            .strip()
        )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": (
                        question
                        + "\nAnswer briefly with only the answer."
                    ),
                },
            ],
        }
    ]

    kwargs = generic_chat_kwargs()

    inputs = (
        processor
        .apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            **kwargs,
        )
    )

    inputs = move_batch(
        inputs
    )

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
    }

    if MODEL_KEY == "minicpm":
        generation_kwargs[
            "downsample_mode"
        ] = "4x"

    generated = eval_model.generate(
        **inputs,
        **generation_kwargs,
    )

    trimmed = [
        out_ids[
            len(in_ids):
        ]
        for in_ids, out_ids
        in zip(
            inputs["input_ids"],
            generated,
        )
    ]

    return (
        processor
        .batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        .strip()
    )


bertscore_scorer = None


def validate_epoch(
    epoch,
):
    # Rank 0 generates predictions and scores them so validation files are written once.
    global bertscore_scorer

    model.module.eval()

    records = []

    for row in tqdm(
        val_df.itertuples(
            index=False
        ),
        total=len(
            val_df
        ),
        desc=f"SFT validation epoch {epoch}",
    ):
        try:
            prediction = predict_answer(
                row.image_path,
                str(
                    row.question
                ),
            )
            error = ""

        except Exception as exc:
            prediction = ""
            error = (
                f"{type(exc).__name__}: "
                f"{str(exc)[:300]}"
            )

        records.append(
            {
                "qa_id": str(
                    row.qa_id
                ),
                "question": str(
                    row.question
                ),
                "answer": str(
                    row.answer
                ),
                "prediction": prediction,
                "error": error,
            }
        )

    predictions = pd.DataFrame(
        records
    )

    valid = (
        predictions.error.eq("")
        & predictions.prediction
        .astype(str)
        .str.strip()
        .ne("")
    )

    evaluated = (
        predictions[
            valid
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    if len(
        evaluated
    ):
        evaluated[
            "exact_match"
        ] = (
            evaluated.prediction
            .map(
                normalize_answer
            )
            .eq(
                evaluated.answer
                .map(
                    normalize_answer
                )
            )
        )

        val_exact_match = float(
            evaluated.exact_match.mean()
        )

        if bertscore_scorer is None:
            print(
                "Loading BERTScore backbone: "
                "roberta-large"
            )

            bertscore_scorer = BERTScorer(
                model_type="roberta-large",
                lang="en",
                device=str(
                    DEVICE
                ),
                batch_size=8,
            )

        bert_p, bert_r, bert_f1 = (
            bertscore_scorer.score(
                evaluated.prediction
                .astype(str)
                .tolist(),
                evaluated.answer
                .astype(str)
                .tolist(),
            )
        )

        evaluated[
            "bertscore_precision"
        ] = bert_p.cpu().numpy()

        evaluated[
            "bertscore_recall"
        ] = bert_r.cpu().numpy()

        evaluated[
            "bertscore_f1"
        ] = bert_f1.cpu().numpy()

        val_bertscore_f1 = float(
            evaluated.bertscore_f1.mean()
        )

    else:
        val_exact_match = float("nan")
        val_bertscore_f1 = float("nan")

    predictions.to_csv(
        RUN_DIR
        / f"val_epoch{epoch}_predictions.csv",
        index=False,
    )

    evaluated.to_csv(
        RUN_DIR
        / f"val_epoch{epoch}_predictions_scored.csv",
        index=False,
    )

    return (
        val_bertscore_f1,
        val_exact_match,
    )


history = []

global_micro_step = 0
optimizer_step = 0

best_bertscore_f1 = float(
    "-inf"
)

best_epoch = None

LATEST_DIR = (
    ADAPTER_DIR
    / "latest"
)

BEST_DIR = (
    ADAPTER_DIR
    / "best"
)

optimizer.zero_grad(
    set_to_none=True
)


for epoch in range(
    1,
    EPOCHS + 1,
):
    # set_epoch changes the sampler order each epoch while keeping all ranks aligned.
    train_sampler.set_epoch(
        epoch
    )

    model.train()

    local_loss_sum = 0.0
    local_examples = 0
    local_skipped = 0

    progress = tqdm(
        train_loader,
        total=len(
            train_loader
        ),
        desc=(
            f"SFT epoch "
            f"{epoch}/{EPOCHS}"
        ),
        disable=not is_main(),
    )

    for batch_rows in progress:
        batch_loss = None
        successful = 0

        for row in batch_rows:
            try:
                inputs = prepare_example(
                    row
                )

                if MODEL_KEY == "minicpm":
                    outputs = model(
                        **inputs,
                        downsample_mode="4x",
                    )
                else:
                    outputs = model(
                        **inputs
                    )

                example_loss = (
                    outputs.loss
                )

                if batch_loss is None:
                    batch_loss = (
                        example_loss
                    )
                else:
                    batch_loss = (
                        batch_loss
                        + example_loss
                    )

                local_loss_sum += float(
                    example_loss
                    .detach()
                    .float()
                    .cpu()
                )

                local_examples += 1
                successful += 1

            except Exception as exc:
                local_skipped += 1

                if is_main():
                    print(
                        f"Skipping QA "
                        f"{row['qa_id']}: "
                        f"{type(exc).__name__}: "
                        f"{str(exc)[:180]}"
                    )

        if successful == 0:
            continue

        batch_loss = (
            batch_loss
            / successful
        )

        scaled_loss = (
            batch_loss
            / GRAD_ACCUM_STEPS
        )

        # Accumulate several micro-batches before updating the shared DDP model.
        scaled_loss.backward()

        global_micro_step += 1

        if (
            global_micro_step
            % GRAD_ACCUM_STEPS
            == 0
        ):
            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for p in model.parameters()
                    if p.requires_grad
                ],
                MAX_GRAD_NORM,
            )

            optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            optimizer_step += 1

            if (
                is_main()
                and optimizer_step
                % SAVE_EVERY_OPT_STEPS
                == 0
            ):
                LATEST_DIR.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                model.module.save_pretrained(
                    LATEST_DIR
                )

        if is_main():
            running_loss = (
                local_loss_sum
                / max(
                    local_examples,
                    1,
                )
            )

            progress.set_postfix(
                loss=f"{running_loss:.4f}",
                opt_step=optimizer_step,
                skipped=local_skipped,
            )

    if (
        global_micro_step
        % GRAD_ACCUM_STEPS
        != 0
    ):
        torch.nn.utils.clip_grad_norm_(
            [
                p
                for p in model.parameters()
                if p.requires_grad
            ],
            MAX_GRAD_NORM,
        )

        optimizer.step()

        optimizer.zero_grad(
            set_to_none=True
        )

        optimizer_step += 1

    stats = torch.tensor(
        [
            local_loss_sum,
            float(
                local_examples
            ),
            float(
                local_skipped
            ),
        ],
        dtype=torch.float64,
        device=DEVICE,
    )

    # Combine per-rank totals so reported epoch metrics cover the full dataset.
    dist.all_reduce(
        stats,
        op=dist.ReduceOp.SUM,
    )

    global_loss = (
        stats[0].item()
        / max(
            stats[1].item(),
            1.0,
        )
    )

    global_skipped = int(
        stats[2].item()
    )

    if is_main():
        epoch_dir = (
            ADAPTER_DIR
            / f"epoch_{epoch}"
        )

        epoch_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.module.save_pretrained(
            epoch_dir
        )

        print(
            f"Epoch {epoch} training complete | "
            f"loss={global_loss:.4f} | "
            f"skipped={global_skipped}"
        )

    # All ranks finish training before rank 0 validates.
    dist.barrier()

    if is_main():
        (
            val_bertscore_f1,
            val_exact_match,
        ) = validate_epoch(
            epoch
        )

        improved = (
            np.isfinite(
                val_bertscore_f1
            )
            and val_bertscore_f1
            > best_bertscore_f1
        )

        if improved:
            best_bertscore_f1 = (
                val_bertscore_f1
            )

            best_epoch = epoch

            if BEST_DIR.exists():
                shutil.rmtree(
                    BEST_DIR
                )

            shutil.copytree(
                ADAPTER_DIR
                / f"epoch_{epoch}",
                BEST_DIR,
            )

            with open(
                BEST_DIR
                / "best_checkpoint.json",
                "w",
            ) as f:
                json.dump(
                    {
                        "best_epoch":
                            best_epoch,
                        "val_bertscore_f1":
                            best_bertscore_f1,
                        "val_exact_match_accuracy":
                            val_exact_match,
                    },
                    f,
                    indent=2,
                )

        history.append(
            {
                "epoch":
                    epoch,
                "mean_train_loss":
                    global_loss,
                "val_bertscore_f1":
                    val_bertscore_f1,
                "val_exact_match_accuracy":
                    val_exact_match,
                "optimizer_steps":
                    optimizer_step,
                "skipped_examples":
                    global_skipped,
                "is_best":
                    bool(
                        improved
                    ),
            }
        )

        pd.DataFrame(
            history
        ).to_csv(
            RUN_DIR
            / "training_history.csv",
            index=False,
        )

        with open(
            RUN_DIR
            / f"epoch_{epoch}_metrics.json",
            "w",
        ) as f:
            json.dump(
                history[-1],
                f,
                indent=2,
            )

        print(
            f"Epoch {epoch} validation | "
            f"BERTScore-F1="
            f"{val_bertscore_f1:.4f} | "
            f"Exact Match="
            f"{val_exact_match:.4f} | "
            f"best_epoch={best_epoch}"
        )

    # Rank 1 may wait here while rank 0 validates. The process
    # group has a 1-hour timeout to avoid the prior NCCL timeout.
    dist.barrier()

    model.train()


# ============================================================
# FINAL SUMMARY
# ============================================================

if is_main():
    if best_epoch is None:
        raise RuntimeError(
            "No valid BERTScore-F1 was produced "
            "during validation."
        )

    final_metrics = {
        "method":
            "SFT",
        "model_key":
            MODEL_KEY,
        "model_id":
            MODEL_ID,
        "seed":
            SEED,
        "world_size":
            WORLD_SIZE,
        "epochs":
            EPOCHS,
        "selection_metric":
            "validation BERTScore-F1",
        "bertscore_model":
            "roberta-large",
        "best_epoch":
            int(
                best_epoch
            ),
        "best_val_bertscore_f1":
            float(
                best_bertscore_f1
            ),
        "learning_rate":
            LEARNING_RATE,
        "per_device_batch_size":
            PER_DEVICE_BATCH_SIZE,
        "grad_accum_steps":
            GRAD_ACCUM_STEPS,
        "effective_batch_size":
            (
                PER_DEVICE_BATCH_SIZE
                * WORLD_SIZE
                * GRAD_ACCUM_STEPS
            ),
        "lora_r":
            LORA_R,
        "lora_alpha":
            LORA_ALPHA,
        "train_examples":
            int(
                len(
                    train_df
                )
            ),
        "val_examples":
            int(
                len(
                    val_df
                )
            ),
        "best_adapter_path":
            str(
                BEST_DIR
            ),
    }

    with open(
        RUN_DIR
        / "final_metrics.json",
        "w",
    ) as f:
        json.dump(
            final_metrics,
            f,
            indent=2,
        )

    print(
        "\nFinal SFT selection:"
    )

    print(
        json.dumps(
            final_metrics,
            indent=2,
        )
    )


dist.barrier()
dist.destroy_process_group()
"""

script_path = Path(
    "/kaggle/working/train_sft_ddp.py"
)

script_path.write_text(
    TRAIN_SCRIPT
)

print(
    "Wrote:",
    script_path
)


Wrote: /kaggle/working/train_sft_ddp.py


## Export notebook settings to the DDP processes

For an initial smoke test use:

```python
TRAIN_LIMIT = 64
VAL_LIMIT = 64
```

Then switch both back to `None` for the full experiment.


In [24]:
import os

os.environ["KAGGLE_DATASET_SLUG"] = KAGGLE_DATASET_SLUG
os.environ["MODEL_KEY"] = MODEL_KEY
os.environ["SEED"] = str(SEED)
os.environ["EPOCHS"] = str(EPOCHS)
os.environ["LEARNING_RATE"] = str(LEARNING_RATE)
os.environ["WEIGHT_DECAY"] = str(WEIGHT_DECAY)
os.environ["PER_DEVICE_BATCH_SIZE"] = str(PER_DEVICE_BATCH_SIZE)
os.environ["GRAD_ACCUM_STEPS"] = str(GRAD_ACCUM_STEPS)
os.environ["MAX_GRAD_NORM"] = str(MAX_GRAD_NORM)
os.environ["LORA_R"] = str(LORA_R)
os.environ["LORA_ALPHA"] = str(LORA_ALPHA)
os.environ["LORA_DROPOUT"] = str(LORA_DROPOUT)
os.environ["MAX_NEW_TOKENS"] = str(MAX_NEW_TOKENS)
os.environ["SAVE_EVERY_OPT_STEPS"] = str(SAVE_EVERY_OPT_STEPS)

os.environ["TRAIN_LIMIT"] = (
    ""
    if TRAIN_LIMIT is None
    else str(TRAIN_LIMIT)
)

os.environ["VAL_LIMIT"] = (
    ""
    if VAL_LIMIT is None
    else str(VAL_LIMIT)
)

# Cleaner tokenizer behavior with multiple worker processes.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("DDP environment configured.")


DDP environment configured.


## Launch training on both T4 GPUs

You should see:

```text
World size: 2
```

near the beginning of the output. That confirms both GPUs are participating.


In [25]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_sft_ddp.py

W1025 12:54:13.154000 430 torch/distributed/run.py:982] 
W1025 12:54:13.154000 430 torch/distributed/run.py:982] *****************************************
W1025 12:54:13.154000 430 torch/distributed/run.py:982] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1025 12:54:13.154000 430 torch/distributed/run.py:982] *****************************************
World size: 2
Train examples: 3479
Validation examples: 753
Effective batch size: 8
Loading model: HuggingFaceTB/SmolVLM-500M-Instruct
processor_config.json: 100%|█████████████████| 68.0/68.0 [00:00<00:00, 246kB/s]
preprocessor_config.json: 100%|███████████████| 486/486 [00:00<00:00, 2.49MB/s]
config.json: 100%|████████████████████████| 7.34k/7.34k [00:00<00:00, 16.6MB/s]
tokenizer_config.json: 100%|██████████████| 28.2k/28.2k [00:00<00:00, 59.7MB/s]
vocab.json: 100%|█████████

## Inspect saved results

The primary validation metric is `val_bertscore_f1`; normalised exact match is retained as `val_exact_match_accuracy`.

In [26]:
from pathlib import Path
import json
import pandas as pd

RUN_DIR = (
    Path("/kaggle/working")
    / "results"
    / "training"
    / "sft"
    / MODEL_KEY
    / f"seed{SEED}"
)

ADAPTER_DIR = (
    Path("/kaggle/working")
    / "models"
    / "sft"
    / MODEL_KEY
    / f"seed{SEED}"
)

BEST_DIR = (
    ADAPTER_DIR
    / "best"
)

print("Run directory:", RUN_DIR)
print("Adapter root:", ADAPTER_DIR)
print("Best adapter:", BEST_DIR)

history_path = (
    RUN_DIR
    / "training_history.csv"
)

metrics_path = (
    RUN_DIR
    / "final_metrics.json"
)

best_checkpoint_path = (
    BEST_DIR
    / "best_checkpoint.json"
)

if history_path.is_file():
    print("\nTraining + validation history:")
    display(
        pd.read_csv(
            history_path
        )
    )

if metrics_path.is_file():
    print("\nFinal metrics:")
    with open(
        metrics_path,
        "r",
    ) as f:
        metrics = json.load(
            f
        )

    print(
        json.dumps(
            metrics,
            indent=2,
        )
    )

if best_checkpoint_path.is_file():
    print("\nBest checkpoint:")
    with open(
        best_checkpoint_path,
        "r",
    ) as f:
        best_info = json.load(
            f
        )

    print(
        json.dumps(
            best_info,
            indent=2,
        )
    )


Run directory: /kaggle/working/results/training/sft/smolvlm/seed42
Adapter root: /kaggle/working/models/sft/smolvlm/seed42
Best adapter: /kaggle/working/models/sft/smolvlm/seed42/best

Training + validation history:


,epoch,mean_train_loss,val_bertscore_f1,val_exact_match_accuracy,optimizer_steps,skipped_examples,is_best
0,1,0.637207,0.968969,0.666667,435,0,True
1,2,0.183731,0.969571,0.674635,870,0,True
2,3,0.073980,0.970684,0.686587,1305,0,True



Final metrics:
{
  "method": "SFT",
  "model_key": "smolvlm",
  "model_id": "HuggingFaceTB/SmolVLM-500M-Instruct",
  "seed": 42,
  "world_size": 2,
  "epochs": 3,
  "selection_metric": "validation BERTScore-F1",
  "bertscore_model": "roberta-large",
  "best_epoch": 3,
  "best_val_bertscore_f1": 0.9706844687461853,
  "learning_rate": 0.0001,
  "per_device_batch_size": 1,
  "grad_accum_steps": 4,
  "effective_batch_size": 8,
  "lora_r": 8,
  "lora_alpha": 16,
  "train_examples": 3479,
  "val_examples": 753,
  "best_adapter_path": "/kaggle/working/models/sft/smolvlm/seed42/best"
}

Best checkpoint:
{
  "best_epoch": 3,
  "val_bertscore_f1": 0.9706844687461853,
  "val_exact_match_accuracy": 0.6865869853917662
}


## Notes

### Effective batch size

With the defaults:

```text
PER_DEVICE_BATCH_SIZE = 1
WORLD_SIZE = 2
GRAD_ACCUM_STEPS = 4
```

the effective batch size is:

```text
1 × 2 × 4 = 8
```

### Kaggle outputs

Kaggle input datasets are read-only. The notebook therefore writes all outputs under:

```text
/kaggle/working/results/
/kaggle/working/models/
```

Save or export the Kaggle notebook output after the run so the trained adapters and metrics are retained.


### Checkpoint layout for the 3-epoch run

```text
/kaggle/working/models/sft/<model>/seed42/
├── epoch_1/
├── epoch_2/
├── epoch_3/
├── latest/
└── best/
    ├── adapter_config.json
    ├── adapter_model.safetensors
    └── best_checkpoint.json
```

`best/` is selected using **validation BERTScore-F1** only.
